# Baseline Data Analysis: Senior School Leadership & CBE (Rural vs Urban)

This notebook runs the full baseline analysis step by step, from loading the KoboToolbox exports to the final tables and charts. Run the cells **in order, top to bottom** (Shift + Enter runs one cell and moves to the next).

**Outputs** are saved to an `outputs` folder next to this notebook:
- `outputs/Baseline_Analysis_Tables.xlsx`: every table in one workbook
- `outputs/figures/*.png`: charts for the report

---

## Step 0. Set up the environment (do this once)

**0.1 Install Python.** Download Python 3.11 or newer from python.org. During installation, tick **"Add python.exe to PATH"**. To check, open a new VS Code terminal and run `py --version`.

**0.2 Install VS Code extensions.** In the Extensions panel (Ctrl + Shift + X), install **Python** and **Jupyter** (both by Microsoft).

**0.3 Put the files in one folder.** This notebook, `requirements.txt` and the two Kobo exports should all be in your *Data Analysis* folder:
```
Data Analysis/
├── Baseline_Analysis.ipynb
├── requirements.txt
├── DATA_COLLECTION_QUESTIONNAIRE_FOR_SCHOOL_LEADERS_Baseline.xlsx
└── DATA_COLLECTION_QUESTIONNAIRE_FOR_SENIOR_SCHOOL_TEACHERS_Baseline.xlsx
```
The file names don't have to match exactly. Step 2 finds any `.xlsx` whose name contains `SCHOOL_LEADERS` or `TEACHERS` plus `Baseline`.

**0.4 Create a virtual environment.** In the VS Code terminal (PowerShell), inside the *Data Analysis* folder, run:
```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
```
If PowerShell says *running scripts is disabled*, run this once, then activate again:
```powershell
Set-ExecutionPolicy -Scope CurrentUser -ExecutionPolicy RemoteSigned
```
When it works, the prompt starts with `(.venv)`.

**0.5 Install the packages.**
```powershell
python -m pip install --upgrade pip
pip install -r requirements.txt
```

**0.6 Select the kernel.** Open this notebook, click **Select Kernel** (top right) → **Python Environments** → choose **.venv**. If VS Code asks to install `ipykernel`, click **Install**.

Now run the cells below in order.

## Step 1. Import libraries and set options

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

# Folders
BASE = Path.cwd()                      # the folder this notebook is in
OUT = BASE / "outputs"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Working folder:", BASE)

## Step 2. Load the two Kobo exports

The code searches the folder for the leaders' and teachers' files, so small differences in file names don't matter.

In [ ]:
def find_file(*keywords):
    hits = [p for p in BASE.glob("*.xlsx")
            if all(k.lower() in p.name.lower() for k in keywords)
            and not p.name.startswith("~$")]          # skip Excel lock files
    if not hits:
        raise FileNotFoundError(f"No .xlsx with {keywords} in {BASE}")
    return hits[0]

leaders_file = find_file("SCHOOL_LEADERS", "Baseline")
teachers_file = find_file("TEACHERS", "Baseline")

L = pd.read_excel(leaders_file)    # school leaders
T = pd.read_excel(teachers_file)   # teachers

print("Leaders :", leaders_file.name, "->", L.shape)
print("Teachers:", teachers_file.name, "->", T.shape)

## Step 3. Inspect the structure

Kobo exports every question, every note (section headings) and the system fields as columns. Notes are completely empty, so we list only columns that hold data.

In [ ]:
def describe_columns(df):
    rows = []
    for c in df.columns:
        s = df[c].dropna()
        if len(s) == 0:
            continue                                  # skip empty note columns
        rows.append({"column": c.split("\n")[0][:90],
                     "non_missing": len(s),
                     "unique": s.astype(str).nunique(),
                     "example": str(s.iloc[0])[:40]})
    return pd.DataFrame(rows)

describe_columns(L).head(40)

In [ ]:
describe_columns(T).head(40)

## Step 4. Basic cleaning

- Keep only consenting respondents.
- Create a short `Location` column (Rural/Urban).
- Compute interview duration in minutes (used in the quality checks).

In [ ]:
CONSENT = "Do you agree to participate in this study?"
for name, df in [("Leaders", L), ("Teachers", T)]:
    print(name, df[CONSENT].value_counts().to_dict())

L = L[L[CONSENT].str.startswith("Yes")].copy()
T = T[T[CONSENT].str.startswith("Yes")].copy()

for df in (L, T):
    df["Location"] = df["A1. Location of the Senior School"]
    df["duration_min"] = (pd.to_datetime(df["end"]) - pd.to_datetime(df["start"])).dt.total_seconds() / 60

print("\nLeaders by location :", L["Location"].value_counts().to_dict())
print("Teachers by location:", T["Location"].value_counts().to_dict())
print("Collection dates: leaders", L["today"].min(), "to", L["today"].max(),
      "| teachers", T["today"].min(), "to", T["today"].max())

## Step 5. Convert Likert answers to numbers

Kobo stores answers as labels such as `"4 = Often"`. We take the leading digit (1 to 5).

In [ ]:
def likert_to_num(series):
    return pd.to_numeric(series.astype(str).str.extract(r"^\s*(\d)\s*=")[0], errors="coerce")

likert_to_num(pd.Series(["5 = Very Often", "2 = Rarely", "3 = Not Sure"]))

## Step 6. Define the scales (groups of items)

Each scale is a list of item columns, matched by their question codes. A scale score is the **mean of its items** (1 to 5).

| Leaders | Teachers |
|---|---|
| B1.1 to B1.10 Management leadership | B1.1 to B1.10 Management leadership (of their leaders) |
| C1 to C10 Instructional leadership | C1 to C8 Instructional leadership |
| D1 to D10 Adaptation of leadership | D1 to D10 Observed adaptation |
| E1.x to E4.x Contextual conditions (4 sub-scales) | E1 to E12 Contextual conditions |
| F1 to F8 Support to teachers | F1 to F12 Self-rated preparedness |
| G1 to G8 Teacher preparedness (leader view) | |

In [ ]:
def cols(df, pattern):
    return [c for c in df.columns if re.match(pattern, c)]

L_SCALES = {
    "Management leadership (B1)":            cols(L, r"B1\.\d+\. "),
    "Instructional leadership (C)":          cols(L, r"C\d+\. I "),
    "Leadership adaptation (D)":             cols(L, r"D([1-9]|10)\. "),
    "E1 Resources & infrastructure":         cols(L, r"E1\.\d\. "),
    "E2 Human resources & workload":         cols(L, r"E2\.\d\. "),
    "E3 Professional & institutional support": cols(L, r"E3\.\d\. "),
    "E4 School & community environment":     cols(L, r"E4\.\d\. "),
    "Support to teachers (F)":               cols(L, r"F\d\. I "),
    "Teacher preparedness - leader view (G)": cols(L, r"G[1-8]\. "),
}
T_SCALES = {
    "Management leadership (B1)":            cols(T, r"B1\.\d+\. "),
    "Instructional leadership (C)":          cols(T, r"C\d\. School"),
    "Leadership adaptation (D)":             cols(T, r"D([1-9]|10)\. "),
    "Contextual conditions (E1-E12)":        cols(T, r"E([1-9]|1[0-2])\. "),
    "Teacher preparedness - self-rated (F1-F12)": cols(T, r"F([1-9]|1[0-2])\. "),
}

# Check the number of items found per scale
for who, sc in [("Leaders", L_SCALES), ("Teachers", T_SCALES)]:
    for k, v in sc.items():
        print(f"{who:9s} {k:45s} {len(v):2d} items")

In [ ]:
# Convert items to numbers and compute scale means
for df, sc in [(L, L_SCALES), (T, T_SCALES)]:
    for items in sc.values():
        for c in items:
            df[c] = likert_to_num(df[c])
    for scale, items in sc.items():
        df[scale] = df[items].mean(axis=1)

L_ALL_ITEMS = sum(L_SCALES.values(), [])
T_ALL_ITEMS = sum(T_SCALES.values(), [])
print("Missing Likert values - leaders:", int(L[L_ALL_ITEMS].isna().sum().sum()),
      "| teachers:", int(T[T_ALL_ITEMS].isna().sum().sum()))

## Step 7. Data-quality checks

We check for:
1. duplicate submissions;
2. very short interviews (under 8 minutes);
3. **straight-lining**, meaning the same answer to almost every item (row SD under 0.3);
4. near-identical response patterns (90% or more of items identical between two records);
5. **school codes whose respondents report different locations**, which matters for a rural–urban comparison.

In [ ]:
from itertools import combinations

def quality_report(df, items, label):
    print(f"===== {label} (n = {len(df)})")
    print("Duplicate _uuid           :", int(df["_uuid"].duplicated().sum()))
    short = df.index[df["duration_min"] < 8].tolist()
    print("Interviews under 8 min    :", short)
    row_sd = df[items].std(axis=1)
    print("Straight-liners (SD < .3) :", df.index[row_sd < 0.3].tolist())
    vals = df[items].to_numpy()
    pairs = [(i, j, round(float((vals[a] == vals[b]).mean()), 2))
             for (a, i), (b, j) in combinations(enumerate(df.index), 2)
             if (vals[a] == vals[b]).mean() >= 0.9]
    print("Near-identical pairs (>=90% same):", pairs)
    print("Missing school_code       :", int(df["school_code"].isna().sum()))

quality_report(L, L_ALL_ITEMS, "Leaders")
quality_report(T, T_ALL_ITEMS, "Teachers")

In [ ]:
# Location consistency within school codes (leaders and teachers combined)
both = pd.concat([L[["school_code", "Location"]].assign(Group="Leader"),
                  T[["school_code", "Location"]].assign(Group="Teacher")],
                 ignore_index=True).dropna()
loc_by_school = pd.crosstab(both["school_code"], both["Location"])
inconsistent = loc_by_school[(loc_by_school > 0).sum(axis=1) > 1]
print("School codes with BOTH Rural and Urban responses:")
display(inconsistent)

# Leaders per school code and principals per code
print("Leader records per school code (top):")
display(L["school_code"].value_counts().head())
principals = L[L["A2. Your current leadership position"].str.startswith("Principal")]
print("Codes with more than one principal:")
display(principals["school_code"].value_counts()[lambda s: s > 1])

print("Schools: leaders", L.school_code.nunique(), "| teachers", T.school_code.nunique(),
      "| union", len(set(L.school_code) | set(T.school_code.dropna())),
      "| with both", len(set(L.school_code) & set(T.school_code.dropna())))

**Decision:** keep all records for the main analysis, use the **respondent-reported location**, and test in Step 15 whether the flagged records or the location inconsistencies change any result.

## Step 8. Reliability of the scales (Cronbach's alpha)

α ≥ .70 is acceptable; ≥ .80 is good.

In [ ]:
def cronbach_alpha(items_df):
    d = items_df.dropna()
    k = d.shape[1]
    return k / (k - 1) * (1 - d.var(ddof=1).sum() / d.sum(axis=1).var(ddof=1))

rel = []
for who, df, sc in [("Leaders", L, L_SCALES), ("Teachers", T, T_SCALES)]:
    for scale, items in sc.items():
        rel.append({"Group": who, "Scale": scale, "Items": len(items),
                    "Alpha": round(cronbach_alpha(df[items]), 2)})
reliability = pd.DataFrame(rel)
reliability

## Step 9. Respondent profile by location

Cross-tabulations give counts and column percentages, with a chi-square test. With only 5 urban leaders, read these p-values with caution.

In [ ]:
def crosstab_pct(df, col):
    t = pd.crosstab(df[col], df["Location"], margins=True, margins_name="Total")
    p = (t / t.loc["Total"] * 100).round(1)
    out = t.astype(str) + " (" + p.astype(str) + "%)"
    out.index.name = col.split("\n")[0]
    ct = pd.crosstab(df[col], df["Location"])
    pval = stats.chi2_contingency(ct)[1] if ct.shape[0] > 1 else np.nan
    return out, pval

L_PROFILE = ["A2. Your current leadership position", "A3. Gender",
             "A4. Number of years in school leadership",
             "A5. Number of years in the current Senior School",
             "A6. What type of Senior School is your school?",
             "A7. Approximate number of learners currently enrolled in the Senior School",
             "A8. Approximately how many teachers currently teach in the Senior School?"]
T_PROFILE = [c for c in T.columns if re.match(r"A[2-8]\. ", c)]

profile_tables = []
for who, df, cs in [("Leaders", L, L_PROFILE), ("Teachers", T, T_PROFILE)]:
    for c in cs:
        tab, p = crosstab_pct(df, c)
        print(f"\n{who} | {c[:80]}   (chi-square p = {p:.3f})")
        display(tab)
        profile_tables.append(tab.reset_index().assign(Group=who, chi2_p=round(p, 3)))

## Step 10. Scale scores and rural vs urban comparison

Likert scale scores are not normally distributed and the urban group is small, so we use the **Mann-Whitney U test**. Effect size is the **rank-biserial r**: positive means urban is higher. As a rough guide, |r| of about .1 is small, .3 medium and .5 large.

In [ ]:
def mann_whitney(a, b):
    a, b = pd.Series(a).dropna(), pd.Series(b).dropna()
    U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    r = 1 - 2 * U / (len(a) * len(b))      # positive = b higher
    return U, p, r

rows = []
for who, df, sc in [("Leaders", L, L_SCALES), ("Teachers", T, T_SCALES)]:
    for scale in sc:
        rur = df.loc[df.Location == "Rural", scale]
        urb = df.loc[df.Location == "Urban", scale]
        U, p, r = mann_whitney(rur, urb)
        rows.append({"Group": who, "Scale": scale,
                     "Mean": df[scale].mean(), "SD": df[scale].std(),
                     "Rural M": rur.mean(), "Rural SD": rur.std(),
                     "Urban M": urb.mean(), "Urban SD": urb.std(),
                     "U": U, "p": p, "r_rb (+ = urban higher)": r})
scales = reliability.merge(pd.DataFrame(rows), on=["Group", "Scale"]).round(2)
scales["p"] = pd.DataFrame(rows)["p"].round(3).values
scales

## Step 11. Item-level descriptives

For every item: mean, SD, % rating 4–5 (often/agree/great extent), % rating 1–2, and the rural and urban means.

In [ ]:
def item_table(df, sc, who):
    out = []
    for scale, items in sc.items():
        for c in items:
            x = df[c]
            out.append({"Group": who, "Scale": scale, "Item": c.split("\n")[0],
                        "Mean": x.mean(), "SD": x.std(),
                        "% 4-5": (x >= 4).mean() * 100, "% 1-2": (x <= 2).mean() * 100,
                        "Rural M": x[df.Location == "Rural"].mean(),
                        "Urban M": x[df.Location == "Urban"].mean()})
    return pd.DataFrame(out).round(2)

items = pd.concat([item_table(L, L_SCALES, "Leaders"), item_table(T, T_SCALES, "Teachers")],
                  ignore_index=True)

# Weakest and strongest leader practices (B1, C, F)
lead_practice = items[(items.Group == "Leaders") & items.Scale.str.contains("B1|\\(C\\)|\\(F\\)")]
print("Lowest-rated leader practices:")
display(lead_practice.nsmallest(5, "Mean")[["Item", "Mean", "% 4-5", "% 1-2"]])
print("Highest-rated leader practices:")
display(lead_practice.nlargest(5, "Mean")[["Item", "Mean", "% 4-5", "% 1-2"]])

In [ ]:
# Teachers' observed change (D) and preparedness (F), sorted
for scale in ["Leadership adaptation (D)", "Teacher preparedness - self-rated (F1-F12)",
              "Contextual conditions (E1-E12)"]:
    print(scale)
    display(items[(items.Group == "Teachers") & (items.Scale == scale)]
            .sort_values("Mean")[["Item", "Mean", "SD", "% 4-5", "% 1-2", "Rural M", "Urban M"]])

## Step 12. Leaders vs teachers: the same practices seen from two sides

1. **Management leadership (B1):** the 10 items are worded identically, so we compare item by item.
2. **Instructional leadership (C)** and **preparedness (G vs F):** the items are matched by meaning (mapping below).

In [ ]:
def compare_items(pairs, label):
    out = []
    for lc, tc in pairs:
        U, p, r = mann_whitney(T[tc], L[lc])          # r positive = leaders higher
        out.append({"Leader item": lc.split("\n")[0][:70], "Teacher item": tc.split(" ")[0],
                    "Leaders": L[lc].mean(), "Teachers": T[tc].mean(),
                    "Gap (L - T)": L[lc].mean() - T[tc].mean(), "p": p, "r": r})
    res = pd.DataFrame(out).round(3)
    print(label); display(res)
    return res

# Scale-level gaps
gap_rows = []
for l_scale, t_scale in [("Management leadership (B1)", "Management leadership (B1)"),
                         ("Instructional leadership (C)", "Instructional leadership (C)"),
                         ("Leadership adaptation (D)", "Leadership adaptation (D)"),
                         ("Teacher preparedness - leader view (G)", "Teacher preparedness - self-rated (F1-F12)")]:
    U, p, r = mann_whitney(T[t_scale], L[l_scale])
    gap_rows.append({"Scale": l_scale, "Leaders": L[l_scale].mean(), "Teachers": T[t_scale].mean(),
                     "Gap (L - T)": L[l_scale].mean() - T[t_scale].mean(), "p": p, "r (+ = leaders higher)": r})
scale_gaps = pd.DataFrame(gap_rows).round(3)
display(scale_gaps)

def code_map(df, pattern):
    return {c.split(".")[0] if not c.startswith("B1.") else ".".join(c.split(".")[:2]): c
            for c in cols(df, pattern)}

Lb, Tb = code_map(L, r"B1\.\d+\. "), code_map(T, r"B1\.\d+\. ")
b1_gaps = compare_items([(Lb[k], Tb[k]) for k in Lb], "B1 management items")

Lc, Tc = code_map(L, r"C\d+\. I "), code_map(T, r"C\d\. School")
C_PAIRS = {"C1": "C1", "C3": "C2", "C4": "C3", "C5": "C4", "C8": "C5", "C7": "C6", "C9": "C7", "C10": "C8"}
c_gaps = compare_items([(Lc[a], Tc[b]) for a, b in C_PAIRS.items()], "Instructional items (matched)")

Lg, Tf = code_map(L, r"G[1-8]\. "), code_map(T, r"F([1-9]|1[0-2])\. ")
G_PAIRS = {"G1": "F1", "G2": "F4", "G3": "F3", "G4": "F6", "G5": "F7", "G6": "F10", "G7": "F9", "G8": "F11"}
prep_gaps = compare_items([(Lg[a], Tf[b]) for a, b in G_PAIRS.items()], "Preparedness items (matched)")

## Step 13. How leadership relates to teacher preparedness

Spearman correlations (ρ). Both measures come from the same respondent, so treat these as **associations, not effects**.

In [ ]:
t_keys = list(T_SCALES)
print("Teachers: correlation matrix")
display(T[t_keys].corr(method="spearman").round(2))

prep_t = "Teacher preparedness - self-rated (F1-F12)"
for k in t_keys[:-1]:
    rho, p = stats.spearmanr(T[k], T[prep_t])
    print(f"{k:35s} rho = {rho:.2f}  p = {p:.4f}")

prep_l = "Teacher preparedness - leader view (G)"
print("\nLeaders: correlation with their view of teacher preparedness")
for k in list(L_SCALES)[:-1]:
    rho, p = stats.spearmanr(L[k], L[prep_l])
    print(f"{k:42s} rho = {rho:.2f}  p = {p:.4f}")

In [ ]:
# Dose-response: number of CBE PD activities vs preparedness
PD_YN = "A6. Have you received professional development or training specifically related to CBE implementation?"
PD_N = "A7. If yes, how many CBE-related professional development activities have you participated in during the past three years?"
order = {"1": 1, "2-3": 2, "4-5": 3, "More than 5": 4}
T["pd_count"] = T[PD_N].astype(str).str.replace(".0", "", regex=False).map(order)

display(T.groupby(PD_N)[prep_t].agg(["count", "mean"]).round(2))
rho, p = stats.spearmanr(T["pd_count"], T[prep_t], nan_policy="omit")
print(f"PD count vs preparedness: rho = {rho:.2f}, p = {p:.4f}")
U, p, r = mann_whitney(T.loc[T[PD_YN] == "No", prep_t], T.loc[T[PD_YN] == "Yes", prep_t])
print(f"Received PD (Yes) vs No: p = {p:.3f}")

## Step 14. Professional learning and PLC readiness

Single-choice items are cross-tabulated. For multi-select items, Kobo creates one 0/1 column per option (`question/option`), so we count the 1s.

In [ ]:
L_CAT = [c for c in L.columns if re.match(r"(D11|E8|G9|H[1-4])\. ", c)]
T_CAT = [c for c in T.columns if re.match(r"(F13|G[1-4]|H[12])\. ", c)]
for who, df, cs in [("Leaders", L, L_CAT), ("Teachers", T, T_CAT)]:
    for c in cs:
        tab, p = crosstab_pct(df, c)
        print(f"\n{who} | {c.splitlines()[0][:90]}")
        display(tab)
        profile_tables.append(tab.reset_index().assign(Group=who, chi2_p=round(p, 3)))

In [ ]:
def multi_select(df, question_prefix):
    opts = [c for c in df.columns if c.startswith(question_prefix + "/")]
    n = int(df[opts[0]].notna().sum())                   # people who answered
    res = []
    for c in opts:
        x = pd.to_numeric(df[c], errors="coerce")
        res.append({"Option": c.split("/", 1)[1], "n": int(x.sum()), "%": round(x.sum() / n * 100, 1),
                    "Rural": int(x[df.Location == "Rural"].sum()),
                    "Urban": int(x[df.Location == "Urban"].sum())})
    out = pd.DataFrame(res).sort_values("n", ascending=False)
    print(f"{question_prefix[:70]}  (n answering = {n})")
    display(out)
    return out

ms_influence = multi_select(L, "D13. What has mainly influenced the changes you have made?")
ms_plc_L = multi_select(L, "H5. Which areas would you most want a PLC to support?")
ms_plc_T = multi_select(T, "H3. Which areas would you most want a PLC to support?")

## Step 15. Coding the open-ended responses into themes

Each answer is checked against a **keyword codebook**. A response can belong to more than one theme. Answers such as "N/a" or "None" are excluded from the base.

> Keyword coding is a first pass. Read the `uncoded` list printed under each table and a sample of coded answers, then adjust the patterns if a theme is missed or over-counted.

In [ ]:
EMPTY = re.compile(r"^\s*(n/?a|na|nan|none\.?|nothing|no|not applicable|no thank.*)\s*$", re.I)

CHALLENGES = {
    "ICT / digital devices & connectivity": r"ict|digital|computer|internet|network|gadget|laptop|tablet|phone|electronic|technolog|electricity|online|smart",
    "Teaching & learning resources / textbooks": r"resource|material|text ?book|books",
    "Facilities, labs & infrastructure": r"practical|facilit|lab|infrastruct|workshop|room",
    "Finance / funding": r"financ|fund|money|budget|expend",
    "Teacher training / retooling & CBE knowledge": r"train|retool|re-tool|capacity|induction|know|skill|understanding|technical (advice|guidance)|guidance|information",
    "Staffing & workload": r"staff|personnel|teachers? (of|with)|human resource|workload|untrained learning|assigned areas|trained teachers",
    "Learner factors (absenteeism, readiness, pathways)": r"abse|learners are|prior knowledge|pathway|enrol|population",
    "Parents / community support": r"parent|communit",
    "Curriculum transition & design": r"curriculum|syllab|8.?4.?4|strand|assessment",
}
CHANGES = {
    "Collaboration & shared/consultative decision-making": r"collab|consult|involv|decision|democratic|team|dialogue|inclusive|sharing|stakeholder|parent|communit",
    "Learner-centred pedagogy & learner voice": r"learner.?cent|learner involvement|student|learners|noisy|interactive|hands.?on|practical|teaching method|methodolog|approach",
    "ICT / technology integration": r"ict|technolog|digital|computer|internet",
    "Resource provision & mobilisation": r"resource|material|facilit|provision|mobilis",
    "Monitoring, supervision & assessment": r"monitor|assess|supervis|performance|progress",
    "Restructuring roles & structures": r"restructur|department|dean|pathway leader|implementation group|section head|employ|realign|plan",
}
SUPPORT = {
    "Training, workshops, seminars & retooling": r"service|sensiti|knowledge|train|seminar|workshop|retool|re-tool|capacity|in-?service|refresher|course|development|induction",
    "ICT equipment & connectivity": r"wifi|ict|digital|computer|internet|laptop|tablet|gadget|projector|smart|technolog|electronic",
    "Teaching & learning resources / textbooks": r"resource|material|text ?book|books|equipment|facilit|lab",
    "Funding / government support": r"fund|financ|government|money|grant|capitation",
    "Collaboration, mentoring & PLCs": r"collab|mentor|peer|benchmark|sharing|plc|share",
    "Technical guidance & supervision": r"guid|support from|supervis|follow|advice|curriculum support|knec|kicd|officer",
    "Staffing": r"staff|employ|recruit|more teachers|personnel|workload",
}

def code_themes(df, col, codebook, label):
    text = df[col].astype(str).str.replace("\n", " ")
    valid = df[col].notna() & ~text.str.match(EMPTY)
    hits = pd.DataFrame({k: text.str.contains(p, case=False, regex=True) & valid
                         for k, p in codebook.items()})
    out = pd.DataFrame({"Theme": hits.columns,
                        "n": hits.sum().values,
                        "% of valid": (hits.sum() / valid.sum() * 100).round(1).values,
                        "Rural": hits[df.Location == "Rural"].sum().values,
                        "Urban": hits[df.Location == "Urban"].sum().values}
                       ).sort_values("n", ascending=False)
    print(f"{label}  (valid answers = {int(valid.sum())})")
    display(out)
    uncoded = text[valid & ~hits.any(axis=1)]
    if len(uncoded):
        print("  uncoded:", list(uncoded)[:10])
    return out.assign(Question=label, Valid_n=int(valid.sum()))

def q(df, start):                       # find a column by the start of its name
    return next(c for c in df.columns if c.startswith(start))

themes = pd.concat([
    code_themes(L, q(L, "E5."), CHALLENGES, "Leaders: main contextual challenge (E5)"),
    code_themes(T, q(T, "E13."), CHALLENGES, "Teachers: main challenge (E13)"),
    code_themes(L, q(L, "D12."), CHANGES, "Leaders: change made (D12)"),
    code_themes(T, q(T, "D11."), CHANGES, "Teachers: change observed in leadership (D11)"),
    code_themes(T, q(T, "I4."), SUPPORT, "Teachers: support that would strengthen preparedness (I4)"),
    code_themes(L, q(L, "H7."), SUPPORT, "Leaders: what would make a PLC useful (H7)"),
    code_themes(T, q(T, "H4."), SUPPORT, "Teachers: what would make a PLC useful (H4)"),
], ignore_index=True)

In [ ]:
# Read actual answers for one theme to check the coding (change the theme / column to explore)
col = q(T, "E13.")
mask = T[col].astype(str).str.contains(CHALLENGES["ICT / digital devices & connectivity"], case=False)
T.loc[mask, [col, "Location"]].head(10)

## Step 16. Sensitivity checks

Do the main conclusions hold if we:
- **(a)** assign each school its majority location (fixing the inconsistent codes), or
- **(b)** drop the flagged records: the leader who answered 5 to every item, one of the identical teacher pair, and teacher forms under 8 minutes?

In [ ]:
# (a) majority location per school code
majority = both.groupby("school_code")["Location"].agg(lambda s: s.value_counts().idxmax())
print("(a) Rural vs urban using majority location per school")
for who, df, sc in [("Leaders", L, L_SCALES), ("Teachers", T, T_SCALES)]:
    d = df.assign(Location=df["school_code"].map(majority)).dropna(subset=["Location"])
    for scale in sc:
        U, p, r = mann_whitney(d.loc[d.Location == "Rural", scale], d.loc[d.Location == "Urban", scale])
        print(f"  {who:8s} {scale:45s} p = {p:.3f}")

In [ ]:
# (b) exclude flagged records (indexes come from the Step 7 output; edit if your data differ)
flag_L = L.index[L[L_ALL_ITEMS].std(axis=1) == 0].tolist()                 # identical answer to every item
vals = T[T_ALL_ITEMS].to_numpy()
identical = [j for (a, i), (b, j) in combinations(enumerate(T.index), 2) if (vals[a] == vals[b]).all()]
flag_T = sorted(set(identical) | set(T.index[T["duration_min"] < 5.5]))
print("Dropping leaders:", flag_L, "| teachers:", flag_T)

L2, T2 = L.drop(index=flag_L), T.drop(index=flag_T)
for who, df, sc in [("Leaders", L2, L_SCALES), ("Teachers", T2, T_SCALES)]:
    for scale in sc:
        U, p, r = mann_whitney(df.loc[df.Location == "Rural", scale], df.loc[df.Location == "Urban", scale])
        print(f"  {who:8s} {scale:45s} mean = {df[scale].mean():.2f}  rural-urban p = {p:.3f}")
U, p, r = mann_whitney(T2[prep_t], L2[prep_l])
print(f"Leader vs teacher preparedness gap still significant? p = {p:.3f}")

## Step 17. Charts

Charts are saved as PNG files in `outputs/figures` for use in Word or PowerPoint.

In [ ]:
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
BLUE, GREY = "#2a6fdb", "#b8b8b0"

# 17.1 Leaders vs teachers on matched scales (dumbbell chart)
d = scale_gaps.copy()
d["Label"] = ["Management leadership", "Instructional leadership", "Leadership adaptation", "Teacher preparedness"]
fig, ax = plt.subplots(figsize=(8, 3.4))
for i, r in d.iloc[::-1].reset_index(drop=True).iterrows():
    ax.plot([r.Teachers, r.Leaders], [i, i], color=GREY, lw=2, zorder=1)
    ax.scatter(r.Teachers, i, s=90, color=GREY, zorder=2, label="Teachers" if i == 0 else None)
    ax.scatter(r.Leaders, i, s=90, color=BLUE, zorder=2, label="School leaders" if i == 0 else None)
    ax.text(r.Leaders + (0.03 if r.Leaders > r.Teachers else -0.03), i, f"{r.Leaders:.2f}",
            va="center", ha="left" if r.Leaders > r.Teachers else "right", color=BLUE)
    ax.text(r.Teachers + (-0.03 if r.Leaders > r.Teachers else 0.03), i, f"{r.Teachers:.2f}",
            va="center", ha="right" if r.Leaders > r.Teachers else "left", color="#555")
ax.set_yticks(range(len(d))); ax.set_yticklabels(d["Label"][::-1])
ax.set_xlim(3.3, 4.4); ax.set_xlabel("Mean score (1-5)")
ax.set_title("Leaders rate their own leadership higher than teachers do,\nbut rate teacher preparedness lower", loc="left", fontsize=11)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig(FIG / "01_leader_vs_teacher.png", dpi=200); plt.show()

In [ ]:
# 17.2 Rural vs urban scale means (shows baseline equivalence)
s = scales.copy()
s["Label"] = s["Group"].str[0] + ": " + s["Scale"]
fig, ax = plt.subplots(figsize=(8, 6))
y = np.arange(len(s))[::-1]
ax.hlines(y, s["Rural M"], s["Urban M"], color=GREY, lw=2)
ax.scatter(s["Rural M"], y, s=60, color=BLUE, label="Rural", zorder=3)
ax.scatter(s["Urban M"], y, s=60, color="#e08a2c", label="Urban", zorder=3)
ax.set_yticks(y); ax.set_yticklabels(s["Label"], fontsize=8)
ax.set_xlim(2.8, 4.5); ax.set_xlabel("Mean score (1-5)")
ax.set_title("No scale differs significantly between rural and urban\nschools at baseline (L = leaders, T = teachers)", loc="left", fontsize=11)
ax.legend(frameon=False, loc="lower left")
plt.tight_layout(); plt.savefig(FIG / "02_rural_vs_urban_scales.png", dpi=200); plt.show()

In [ ]:
# 17.3 Main challenges: leaders vs teachers (% of respondents mentioning each theme)
ch = themes[themes.Question.str.contains("challenge")]
pivot = ch.pivot_table(index="Theme", columns="Question", values="% of valid")
pivot.columns = ["Leaders" if c.startswith("Leaders") else "Teachers" for c in pivot.columns]
pivot = pivot.sort_values("Teachers")
fig, ax = plt.subplots(figsize=(8, 4.5))
yy = np.arange(len(pivot))
ax.barh(yy + 0.2, pivot["Teachers"], height=0.4, color=GREY, label="Teachers")
ax.barh(yy - 0.2, pivot["Leaders"], height=0.4, color=BLUE, label="Leaders")
for i, (t, l) in enumerate(zip(pivot["Teachers"], pivot["Leaders"])):
    ax.text(t + 0.8, i + 0.2, f"{t:.0f}%", va="center", fontsize=8)
    ax.text(l + 0.8, i - 0.2, f"{l:.0f}%", va="center", fontsize=8, color=BLUE)
ax.set_yticks(yy); ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_xlabel("% of respondents mentioning the theme")
ax.set_title("Teachers cite resources, facilities and ICT;\nleaders add teacher retooling", loc="left", fontsize=11)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig(FIG / "03_challenge_themes.png", dpi=200); plt.show()

In [ ]:
# 17.4 Preparedness by number of CBE PD activities
g = T.groupby("pd_count")[prep_t].agg(["mean", "count"])
labels = {1: "1", 2: "2-3", 3: "4-5", 4: "More than 5"}
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([labels[i] for i in g.index], g["mean"], color=BLUE, width=0.6)
for i, (m, n) in enumerate(zip(g["mean"], g["count"])):
    ax.text(i, m + 0.08, f"{m:.2f}\n(n={n})", ha="center", fontsize=8)
ax.set_ylim(0, 5); ax.set_ylabel("Self-rated preparedness (1-5)")
ax.set_xlabel("CBE PD activities attended in past 3 years")
ax.set_title("Preparedness rises with the number of CBE PD activities", loc="left", fontsize=11)
plt.tight_layout(); plt.savefig(FIG / "04_pd_dose_preparedness.png", dpi=200); plt.show()

## Step 18. Export all tables to Excel

In [ ]:
out_file = OUT / "Baseline_Analysis_Tables.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as xw:
    scales.to_excel(xw, sheet_name="A1 Scales", index=False)
    items.to_excel(xw, sheet_name="A2 Items", index=False)
    pd.concat([scale_gaps.assign(Level="Scale"), b1_gaps.assign(Level="B1 items"),
               c_gaps.assign(Level="C items"), prep_gaps.assign(Level="Preparedness items")]
              ).to_excel(xw, sheet_name="A3 Leader-teacher gaps", index=False)
    row = 0
    for tab in profile_tables:
        tab.to_excel(xw, sheet_name="A4 Categorical", startrow=row, index=False)
        row += len(tab) + 3
    pd.concat([ms_influence.assign(Question="Leaders D13 influences"),
               ms_plc_L.assign(Question="Leaders H5 PLC areas"),
               ms_plc_T.assign(Question="Teachers H3 PLC areas")]
              ).to_excel(xw, sheet_name="A5 Multi-select", index=False)
    themes.to_excel(xw, sheet_name="A6 Themes", index=False)
    T[t_keys].corr(method="spearman").round(2).to_excel(xw, sheet_name="A7 Correlations T")
    L[list(L_SCALES)].corr(method="spearman").round(2).to_excel(xw, sheet_name="A8 Correlations L")
print("Saved:", out_file)
print("Figures:", [p.name for p in FIG.glob("*.png")])

---
### Done

**Before the endline round, reuse this notebook:** put the endline exports in the folder, change `"Baseline"` to `"Endline"` in Step 2, and run everything again. To compare rounds, stack the two cleaned datasets with a `round` column and compare scale means by round and location (e.g. Mann-Whitney per round, or a difference-in-differences on school-level means once school codes are cleaned).